In [ ]:
# #This module uses a VGG16 CNN to classify white blood cell types

In [2]:
import os
import numpy as np
from pathlib import Path
import random
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
import sys

from model import VGG16
from train import train_one_epoch, validate
from dataset import get_loaders
import gdown
from datetime import datetime

In [3]:
# Choose your datset

#dataset_dir = Path.cwd() / "dataset_5classes/train"
dataset_dir = Path.cwd() / "FullDataset"


In [4]:
# =========================================================
# 1. Settings
# =========================================================

# Training settings
batch_size = 32
num_epochs = 10
learning_rate = 1e-4
image_size = 224
val_split = 0.2
random_seed = 42

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


#==================================================
# 2. Load the Dataset!
#==================================================
train_loader, val_loader, class_names, num_classes = get_loaders(
    dataset_dir, batch_size, image_size, val_split, random_seed
)

print(f"Train images: {len(train_loader.dataset)}")
print(f"Validation images: {len(val_loader.dataset)}")

# =========================================================
# 3. VGG16 Model
# =========================================================
model = VGG16(num_classes=num_classes).to(device)
print(model)


# =========================================================
# 4. Loss and optimizer
# =========================================================
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

# =========================================================
# 5. Training loop
# =========================================================
best_val_acc = 0.0

print(datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
for epoch in range(num_epochs):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = validate(model, val_loader, criterion, device)

    print(
        f"Epoch [{epoch+1}/{num_epochs}] "
        f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
        f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}"
    )

    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "best_vgg16.pth")
        print("Saved best model to best_vgg16.pth")


# =========================================================
# 6. Save final model
# =========================================================
torch.save(model.state_dict(), "final_vgg16.pth")
print("Training complete.")
print("Saved final model to final_vgg16.pth")
print(datetime.now().strftime("%Y-%m-%d %H:%M:%S"))

Using device: cuda
Train images: 25568
Validation images: 6391
VGG16(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(inplace=True)
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): ReLU(inplace=True)
    (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): ReLU(inplace=True)
    (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): ReLU(inplace=True)
    (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (15): ReLU(inplace=True)
 